In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from PyDI.io import load_parquet

amazon_books = load_parquet(
    DATA_DIR / "amazon_new.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads_new.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks_new.parquet",
  name="metabooks"
)

In [3]:
# datasets = [amazon_books, goodreads, metabooks]
# names = ["Amazon Books", "Goodreads", "Metabooks"]

# total_records = sum(len(df) for df in datasets)
# print(f"Total records across all datasets: {total_records:,}")

In [4]:
# from PyDI.profiling import DataProfiler

# # Initialize the DataProfiler
# profiler = DataProfiler()

# for df, name in zip(datasets, names):
#     profile = profiler.summary(df)

# display(profile)

In [5]:
import logging

LOGS = OUTPUT_DIR / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler(LOGS / 'pydi.log'),
          logging.StreamHandler()
      ],
    force=True
)

In [10]:
amazon_books.head(3)

,id,title,author,publish_year,publisher,isbn_clean,title_norm,author_norm
0,amazon_1,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,0195153448,classical mythology,mark p o morford
1,amazon_2,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,0002005018,clara callan,richard bruce wright
2,amazon_3,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,0060973129,decision in normandy,carlo deste


In [11]:
goodreads.head(1)

,id,title,author,rating,num_ratings,language,genres,book_format,edition,pages,publisher,publish_year,price,isbn_clean,title_norm,author_norm
0,goodreads_1,The Hunger Games,Suzanne Collins,4.33,6376780,English,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...",Hardcover,First Edition,374,Scholastic Press,2008,5.09,0439023483,the hunger games,suzanne collins


In [12]:
metabooks.head(1)

,id,title,author,rating,rating_number,language,genres,publisher,page_count,price,publish_year,isbn_clean,title_norm,author_norm
0,metabooks_1,Alvis: The Story of the Red Triangle,Kenneth Day,4.1,3,English,"['Engineering', 'Transportation', 'Automotive']",Haynes Pubns,400,112.300003,<NA>,1844255247,alvis the story of the red triangle,kenneth day


## Entity Matching
### Standard Blocker

In [14]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import RuleBasedMatcher
import numpy as np

/Users/abd/Developer/team_project_data_integration/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

BLOCK_EVAL_DIR.mkdir(parents=True, exist_ok=True)
CORR_DIR.mkdir(parents=True, exist_ok=True)

In [16]:
blocker_m2a = StandardBlocker(
    metabooks, amazon_books,
    on=['isbn_clean'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

standard_candidates_m2a = blocker_m2a.materialize()

blocker_m2g = StandardBlocker(
    metabooks, goodreads,
    on=['isbn_clean'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

standard_candidates_m2g = blocker_m2g.materialize()

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 534656 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 271045 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 27598 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/abd/Developer/team_project_data_integration/books-integration/output/blocking_evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 534656 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 43034 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 6058 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/abd/Developer/

In [17]:
comparators_m2a = [
    # Title similarity
    StringComparator(
        column='title_norm',
        similarity_function='jaccard', 
    ),
    
    # author similarity
    StringComparator(
        column='author_norm',
        similarity_function='jaccard', 
        preprocess=str.lower,
    ),

    # publish year similarity
    NumericComparator(
        column='publish_year',
        max_difference=1,
    ),
]

matcher = RuleBasedMatcher()

st_corr_m2a = matcher.match(
    df_left=metabooks,
    df_right=amazon_books, 
    candidates=blocker_m2a,
    comparators=comparators_m2a,
    weights=[0.4, 0.4, 0.2], 
    threshold=0.7,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 535159 x 271360 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 535159 x 271360 elements after 0:00:0.350; 27776 blocked pairs (reduction ratio: 0.9999998087325626)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:5.778; found 18892 correspondences.


In [18]:
comparators_m2g = comparators_m2a + [

    # genres similarity
    StringComparator(
        column='genres',
        similarity_function='jaccard',
        preprocess=str.lower,
        list_strategy='concatenate'
    )
    
]


st_corr_m2g = matcher.match(
    df_left=metabooks,
    df_right=goodreads, 
    candidates=blocker_m2g,
    comparators=comparators_m2g,
    weights=[0.4, 0.3, 0.1, 0.2], 
    threshold=0.7,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 535159 x 52478 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 535159 x 52478 elements after 0:00:0.120; 6108 blocked pairs (reduction ratio: 0.9999997825101871)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:1.822; found 1293 correspondences.


In [19]:
st_corr_m2a.to_csv(CORR_DIR / "StandardBlocker-Corr-MA.csv", index=False)
st_corr_m2g.to_csv(CORR_DIR / "StandardBlocker-Corr-MG.csv", index=False)

### Evaluate Standard Blocker

In [20]:
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator

test_set_ma = load_csv(
    MLDS_DIR / "test_MA.csv", 
    name="test_entity_matching",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

st_eval_results_ma = EntityMatchingEvaluator.evaluate_matching(
    correspondences=st_corr_m2a,
    test_pairs=test_set_ma,
    out_dir=debug_output_dir
)

display(st_eval_results_ma)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  27
[INFO ] root -   True Negatives:  159
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 14
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.930
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.659
[INFO ] root -   F1-Score:  0.794


{'precision': 1.0,
 'recall': 0.6585365853658537,
 'f1': 0.7941176470588235,
 'accuracy': 0.93,
 'true_positives': 27,
 'false_positives': 0,
 'false_negatives': 14,
 'true_negatives': 159,
 'threshold_used': 0.0,
 'total_correspondences': 18892,
 'filtered_correspondences': 18892,
 'evaluation_timestamp': '2025-11-03T00:37:42.322184',
 'output_files': ['/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [22]:
test_set_mg = load_csv(
    MLDS_DIR / "test_MG.csv", 
    name="test_entity_matching",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

st_eval_results_mg = EntityMatchingEvaluator.evaluate_matching(
    correspondences=st_corr_m2g,
    test_pairs=test_set_mg,
    out_dir=debug_output_dir
)

display(st_eval_results_mg)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  9
[INFO ] root -   True Negatives:  156
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 35
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.825
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.205
[INFO ] root -   F1-Score:  0.340


{'precision': 1.0,
 'recall': 0.20454545454545456,
 'f1': 0.339622641509434,
 'accuracy': 0.825,
 'true_positives': 9,
 'false_positives': 0,
 'false_negatives': 35,
 'true_negatives': 156,
 'threshold_used': 0.0,
 'total_correspondences': 1293,
 'filtered_correspondences': 1293,
 'evaluation_timestamp': '2025-11-03T00:38:03.438138',
 'output_files': ['/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

### Sorted Neighbourhood Blocker

In [23]:
from PyDI.entitymatching import SortedNeighbourhoodBlocker

sn_blocker_m2a = SortedNeighbourhoodBlocker(
    metabooks, amazon_books,
    key='title_norm',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2a = sn_blocker_m2a.materialize()


sn_blocker_m2g = SortedNeighbourhoodBlocker(
    metabooks, goodreads,
    key='title_norm',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2g = sn_blocker_m2g.materialize()

[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created sorted neighbourhood with window size 20
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created 1 sorted sequence from 806519 records
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - Debug results written to file: /Users/abd/Developer/team_project_data_integration/books-integration/output/blocking_evaluation/debugResultsBlocking_SortedNeighbourhoodBlocker.csv
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created sorted neighbourhood with window size 20
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created 1 sorted sequence from 587637 records
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - Debug results written to file: /Users/abd/Developer/team_project_data_integration/books-integration/output/bl

In [24]:
sn_corr_m2a = matcher.match(
    df_left=metabooks,
    df_right=amazon_books, 
    candidates=sn_blocker_m2a,
    comparators=comparators_m2a,
    weights=[0.4, 0.4, 0.2], 
    threshold=0.7,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 535159 x 271360 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 535159 x 271360 elements after 0:00:3.562; 6731058 blocked pairs (reduction ratio: 0.9999536494738233)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:1465.935; found 52873 correspondences.


In [25]:
sn_corr_m2g = matcher.match(
    df_left=metabooks,
    df_right=goodreads, 
    candidates=sn_blocker_m2g,
    comparators=comparators_m2g,
    weights=[0.4, 0.3, 0.1, 0.2], 
    threshold=0.7,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 535159 x 52478 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 535159 x 52478 elements after 0:00:1.710; 1797032 blocked pairs (reduction ratio: 0.9999360124175761)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:489.620; found 5925 correspondences.


In [26]:
sn_corr_m2a.to_csv(CORR_DIR / "SNBlocker-Corr-MA.csv", index=False)
sn_corr_m2g.to_csv(CORR_DIR / "SNBlocker-Corr-MG.csv", index=False)

### Evaluate Sorted Neighbourhood Blocker

In [27]:
sn_eval_results_ma = EntityMatchingEvaluator.evaluate_matching(
    correspondences=sn_corr_m2a,
    test_pairs=test_set_ma,
    out_dir=debug_output_dir
)

display(sn_eval_results_ma)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  26
[INFO ] root -   True Negatives:  159
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 15
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.925
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.634
[INFO ] root -   F1-Score:  0.776


{'precision': 1.0,
 'recall': 0.6341463414634146,
 'f1': 0.7761194029850745,
 'accuracy': 0.925,
 'true_positives': 26,
 'false_positives': 0,
 'false_negatives': 15,
 'true_negatives': 159,
 'threshold_used': 0.0,
 'total_correspondences': 52873,
 'filtered_correspondences': 52873,
 'evaluation_timestamp': '2025-11-03T01:10:57.643649',
 'output_files': ['/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [28]:
sn_eval_results_mg = EntityMatchingEvaluator.evaluate_matching(
    correspondences=sn_corr_m2g,
    test_pairs=test_set_mg,
    out_dir=debug_output_dir
)

display(sn_eval_results_mg)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  9
[INFO ] root -   True Negatives:  156
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 35
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.825
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.205
[INFO ] root -   F1-Score:  0.340


{'precision': 1.0,
 'recall': 0.20454545454545456,
 'f1': 0.339622641509434,
 'accuracy': 0.825,
 'true_positives': 9,
 'false_positives': 0,
 'false_negatives': 35,
 'true_negatives': 156,
 'threshold_used': 0.0,
 'total_correspondences': 5925,
 'filtered_correspondences': 5925,
 'evaluation_timestamp': '2025-11-03T01:10:59.220068',
 'output_files': ['/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

### Token Blocker

In [ ]:
from PyDI.entitymatching import TokenBlocker


token_blocker_m2a = TokenBlocker(
    metabooks, amazon_books,
    column='title_norm',
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id',
    ngram_size=2,
    ngram_type='character'
)
token_candidates_m2a = token_blocker_m2a.materialize()


token_blocker_m2g = TokenBlocker(
    metabooks, goodreads,
    column='title_norm',
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id',
    ngram_size=2,
    ngram_type='character'
)
token_candidates_m2g = token_blocker_m2g.materialize()

[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 1290 token keys for first dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 1268 token keys for second dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 1266 blocks from token keys
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - Debug results written to file: /Users/abd/Developer/team_project_data_integration/books-integration/output/blocking_evaluation/debugResultsBlocking_TokenBlocker.csv


In [ ]:
token_corr_m2a = matcher.match(
    df_left=metabooks,
    df_right=amazon_books, 
    candidates=token_blocker_m2a,
    comparators=comparators_m2a,
    weights=[0.4, 0.4, 0.2], 
    threshold=0.7,
    id_column='id'
)

In [ ]:
token_corr_m2g = matcher.match(
    df_left=metabooks,
    df_right=goodreads, 
    candidates=token_blocker_m2g,
    comparators=comparators_m2g,
    weights=[0.4, 0.3, 0.1, 0.2], 
    threshold=0.7,
    id_column='id'
)

In [ ]:
token_corr_m2a.to_csv(CORR_DIR / "TokenBlocker-Corr-MA.csv", index=False)
token_corr_m2g.to_csv(CORR_DIR / "TokenBlocker-Corr-MG.csv", index=False)

### Evaluate Token Blocker

In [ ]:
token_eval_results_ma = EntityMatchingEvaluator.evaluate_matching(
    correspondences=token_corr_m2a,
    test_pairs=test_set_ma,
    out_dir=debug_output_dir
)

display(token_eval_results_ma)

In [ ]:
token_eval_results_mg = EntityMatchingEvaluator.evaluate_matching(
    correspondences=token_corr_m2g,
    test_pairs=test_set_mg,
    out_dir=debug_output_dir
)

display(token_eval_results_mg)

### Embedding Blocker

In [ ]:
from PyDI.entitymatching import EmbeddingBlocker

embedding_blocker_m2a = EmbeddingBlocker(
    metabooks, amazon_books,
    text_cols=['title_norm', 'author_norm'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
embedding_candidates_m2a = embedding_blocker_m2a.materialize()


embedding_blocker_m2g = EmbeddingBlocker(
    metabooks, goodreads,
    text_cols=['title_norm', 'author_norm'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
embedding_candidates_m2g = embedding_blocker_m2g.materialize()

In [ ]:
em_corr_m2a = matcher.match(
    df_left=metabooks,
    df_right=amazon_books, 
    candidates=embedding_blocker_m2a,
    comparators=comparators_m2a,
    weights=[0.4, 0.4, 0.2], 
    threshold=0.7,
    id_column='id'
)

In [ ]:
em_corr_m2g = matcher.match(
    df_left=metabooks,
    df_right=goodreads, 
    candidates=embedding_blocker_m2g,
    comparators=comparators_m2g,
    weights=[0.4, 0.3, 0.1, 0.2], 
    threshold=0.7,
    id_column='id'
)

In [ ]:
em_corr_m2a.to_csv(CORR_DIR / "EmbeddingBlocker-Corr-MA.csv", index=False)
em_corr_m2g.to_csv(CORR_DIR / "EmbeddingBlocker-Corr-MG.csv", index=False)

### Evaluate Embedding Blocker

In [ ]:
em_eval_results_ma = EntityMatchingEvaluator.evaluate_matching(
    correspondences=em_corr_m2a,
    test_pairs=test_set_ma,
    out_dir=debug_output_dir
)

display(em_eval_results_ma)

In [ ]:
em_eval_results_mg = EntityMatchingEvaluator.evaluate_matching(
    correspondences=em_corr_m2g,
    test_pairs=test_set_mg,
    out_dir=debug_output_dir
)

display(em_eval_results_mg)